In [1]:
pip install pdfplumber pandas openpyxl


   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   ----------- ---------------------------- 1.6/5.6 MB 8.4 MB/s eta 0:00:01
   ---------------------- ----------------- 3.1/5.6 MB 7.4 MB/s eta 0:00:01
   ----------------------------- ---------- 4.2/5.6 MB 6.5 MB/s eta 0:00:01
   ------------------------------------- -- 5.2/5.6 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 5.6/5.6 MB 6.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   -------------------- ------------------- 1.6/3.1 MB 8.4 MB/s eta 0:00:01
   ------------------------------------- -- 2.9/3.1 MB 7.3 MB/s eta 0:00:01
   ---------------------------------------- 3.1/3.1 MB 6.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pdfplumber
import pandas as pd
from datetime import datetime
import re

# PDF file paths
pdf_files = [
    "Income list - Shfaqat Rana Muhammad Muneeb - 2024.01.01-2024.12.31.pdf",
    "Income list - Shfaqat Rana Muhammad Muneeb - 2025.01.01-2025.10.31.pdf"
]

records = []

date_pattern = re.compile(r"\d{4}\.\d{2}\.\d{2}")

for pdf_file in pdf_files:
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue

            for line in text.split("\n"):
                parts = line.split()

                for i, part in enumerate(parts):
                    if date_pattern.fullmatch(part):
                        try:
                            date = datetime.strptime(part, "%Y.%m.%d")

                            # find the numeric net amount after date
                            for j in range(i + 1, min(i + 6, len(parts))):
                                try:
                                    amount = float(parts[j].replace(",", ""))
                                    records.append({
                                        "Year": date.year,
                                        "Month": date.strftime("%B %Y"),
                                        "Gross Income": amount
                                    })
                                    break
                                except:
                                    continue
                        except:
                            pass

# Create DataFrame
df = pd.DataFrame(records)

# Monthly aggregation
monthly = df.groupby(["Year", "Month"], as_index=False).sum()

# Tax & Net calculation
monthly["Tax (40%)"] = monthly["Gross Income"] * 0.40
monthly["Net Income"] = monthly["Gross Income"] - monthly["Tax (40%)"]

# Yearly totals
yearly_totals = (
    monthly.groupby("Year", as_index=False)
    [["Gross Income", "Tax (40%)", "Net Income"]]
    .sum()
)
yearly_totals["Month"] = "TOTAL"

# Combine monthly + totals
final_df = pd.concat([monthly, yearly_totals], ignore_index=True)

# Export to Excel
final_df.to_excel("Monthly_Income_2024_2025.xlsx", index=False)

print("Excel file created successfully!")


BIZONYLAT LISTA SORONKÉNT Shfaqat Rana Muhammad Muneeb (90514124-1-42) - 2025
Dátum alapja: Pénzügyi teljesítés dátuma ˅ Szűrő: sorszam x ; biz.kelt x ; telj.kelt x ; ne(cid:425)o x ; afa x ;
Dátum szerin(cid:415) szűkítés: 2024.01.01-2024.12.31 bru(cid:425)ó x ; fiz.mod x ; kiegyenlitve x ; kiegy.kelt x ;
Számla Számla Számla Számla Számla
Bizonylat Bizonylat Teljesítés Fizetési
nettó ÁFA Bruttó kiegyenlítve kiegyenlítés
sorszáma: kelte: kelte: mód:
végösszege: tartalma: végösszege (igen/nem): kelte:
HU9051412411100000001 2024.11.04 2024.11.10 122627 0 122627 n/a igen 2024.11.04
HU9051412411100000002 2024.11.08 2024.11.17 218121 0 218121 n/a igen 2024.11.08
HU9051412411100000003 2024.11.18 2024.11.25 207106 0 207106 n/a igen 2024.11.18
HU9051412411100000004 2024.11.25 2024.12.02 196080 0 196080 n/a igen 2024.11.25
HU9051412411100000005 2024.12.02 2024.12.10 311370 0 311370 n/a igen 2024.12.02
SHFRA-2024-53 2024.12.07 2024.12.07 749 0 749 Készpénz igen 2024.12.07
HU9051412411100000006 

In [3]:
final_df


,Year,Month,Gross Income,Tax (40%),Net Income
0,2024,December 2024,2395819.0,958327.6,1437491.4
1,2024,November 2024,1291788.0,516715.2,775072.8
2,2025,April 2025,2994660.0,1197864.0,1796796.0
3,2025,August 2025,2415252.0,966100.8,1449151.2
4,2025,February 2025,1596282.0,638512.8,957769.2
5,2025,January 2025,1541969.0,616787.6,925181.4
6,2025,July 2025,3435820.0,1374328.0,2061492.0
7,2025,June 2025,2861956.0,1144782.4,1717173.6
8,2025,March 2025,2372638.0,949055.2,1423582.8
9,2025,May 2025,2982710.0,1193084.0,1789626.0


In [4]:
df

,Year,Month,Gross Income
0,2024,November 2024,122627.0
1,2024,November 2024,122627.0
2,2024,November 2024,218121.0
3,2024,November 2024,218121.0
4,2024,November 2024,207106.0
...,...,...,...
10167,2025,September 2025,699.0
10168,2025,October 2025,575827.0
10169,2025,October 2025,575827.0
10170,2025,October 2025,468371.0


In [5]:
records

[{'Year': 2024, 'Month': 'November 2024', 'Gross Income': 122627.0},
 {'Year': 2024, 'Month': 'November 2024', 'Gross Income': 122627.0},
 {'Year': 2024, 'Month': 'November 2024', 'Gross Income': 218121.0},
 {'Year': 2024, 'Month': 'November 2024', 'Gross Income': 218121.0},
 {'Year': 2024, 'Month': 'November 2024', 'Gross Income': 207106.0},
 {'Year': 2024, 'Month': 'November 2024', 'Gross Income': 207106.0},
 {'Year': 2024, 'Month': 'November 2024', 'Gross Income': 196080.0},
 {'Year': 2024, 'Month': 'December 2024', 'Gross Income': 196080.0},
 {'Year': 2024, 'Month': 'December 2024', 'Gross Income': 311370.0},
 {'Year': 2024, 'Month': 'December 2024', 'Gross Income': 311370.0},
 {'Year': 2024, 'Month': 'December 2024', 'Gross Income': 749.0},
 {'Year': 2024, 'Month': 'December 2024', 'Gross Income': 749.0},
 {'Year': 2024, 'Month': 'December 2024', 'Gross Income': 13799.0},
 {'Year': 2024, 'Month': 'December 2024', 'Gross Income': 13799.0},
 {'Year': 2024, 'Month': 'December 2024', 